In [7]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/D11_InterIIT"

import os
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from pulp import LpProblem, LpVariable, LpMaximize, lpSum, value, PULP_CBC_CMD


df = pd.read_csv(f"{BASE}/cleaned_dataset.csv", low_memory=False)
df['date']     = pd.to_datetime(df['date'])
df['match_id'] = df['match_id'].astype(str)


TARGET = 'fantasy_points'

DROP_COLS = [
    'match_id', 'date', 'venue', 'match_type',
    'team', 'opposition', 'player', 'role', 'dismissal_kind'
]

LEAKY_COLS = [
    'balls_per_boundary', 'boundary_percent', 'boundary_runs',
    'bowling_sr', 'dot_ball_percent', 'dots', 'extras',
    'fielding_points_raw', 'fifty_plus', 'five_wicket_haul',
    'hundred', 'is_out', 'maiden_overs', 'overs', 'three_wicket_haul'
]

FEATURE_COLS = [
    col for col in df.columns
    if col not in DROP_COLS + [TARGET] + LEAKY_COLS
]

ROLES = ['BAT', 'BOWL', 'AR', 'WK']



CUTOFF    = '2024-06-30'
VAL_START = '2024-05-01'

train_df = df[df['date'] <= pd.Timestamp(CUTOFF)].copy()
test_df  = df[df['date'] >  pd.Timestamp(CUTOFF)].copy()

assert train_df['date'].max() <= pd.Timestamp(CUTOFF), "LEAKAGE: training data past cutoff"
assert len(test_df[test_df['date'] <= pd.Timestamp(CUTOFF)]) == 0, "Test set contaminated"

local_train = train_df[train_df['date'] <  VAL_START]
local_val   = train_df[train_df['date'] >= VAL_START]

print(f"local_train : {len(local_train):,} rows")
print(f"local_val   : {len(local_val):,} rows")
print(f"train_full  : {len(train_df):,} rows")
print(f"test        : {len(test_df):,} rows")



BEST_PARAMS = {
    'BAT': {
        'n_estimators': 408, 'max_depth': 8,
        'learning_rate': 0.012925, 'subsample': 0.7776,
        'colsample_bytree': 0.6359, 'min_child_weight': 6,
        'random_state': 42, 'n_jobs': 1
    },
    'BOWL': {
        'n_estimators': 437, 'max_depth': 6,
        'learning_rate': 0.018627, 'subsample': 0.7655,
        'colsample_bytree': 0.7753, 'min_child_weight': 6,
        'random_state': 42, 'n_jobs': 1
    },
    'AR': {
        'n_estimators': 378, 'max_depth': 7,
        'learning_rate': 0.011623, 'subsample': 0.8169,
        'colsample_bytree': 0.6415, 'min_child_weight': 2,
        'random_state': 42, 'n_jobs': 1
    },
    'WK': {
        'n_estimators': 674, 'max_depth': 6,
        'learning_rate': 0.014482, 'subsample': 0.8677,
        'colsample_bytree': 0.9286, 'min_child_weight': 9,
        'random_state': 42, 'n_jobs': 1
    },
}


print("\nValidation MAEs (local split):")
val_maes = {}

for role in ROLES:
    r_train = local_train[local_train['role'] == role]
    r_val   = local_val[local_val['role'] == role]

    model = XGBRegressor(**BEST_PARAMS[role])
    model.fit(r_train[FEATURE_COLS].fillna(0), r_train[TARGET])

    preds          = model.predict(r_val[FEATURE_COLS].fillna(0))
    mae            = mean_absolute_error(r_val[TARGET], preds)
    val_maes[role] = mae
    print(f"  {role:<5} MAE: {mae:.2f}")

total_rows   = sum(len(local_val[local_val['role'] == r]) for r in ROLES)
weighted_mae = sum(val_maes[r] * len(local_val[local_val['role'] == r]) for r in ROLES) / total_rows
print(f"  Weighted MAE: {weighted_mae:.2f}")



print("\nTraining final models on full training set:")
final_models = {}

for role in ROLES:
    role_data = train_df[train_df['role'] == role]
    model     = XGBRegressor(**BEST_PARAMS[role])
    model.fit(role_data[FEATURE_COLS].fillna(0), role_data[TARGET])
    final_models[role] = model
    print(f"  {role:<5} trained on {len(role_data):,} rows")



print("\nTop 10 features per role:")
for role in ROLES:
    importance = pd.DataFrame({
        'feature':    FEATURE_COLS,
        'importance': final_models[role].feature_importances_
    }).sort_values('importance', ascending=False)
    print(f"\n{role}:")
    print(importance.head(10).to_string(index=False))



os.makedirs(f'{BASE}/src/model_artifacts', exist_ok=True)

artifact = {
    'models':          final_models,
    'feature_cols':    FEATURE_COLS,
    'training_cutoff': CUTOFF,
}

with open(f'{BASE}/src/model_artifacts/ProductUI_Model', 'wb') as f:
    pickle.dump(artifact, f)

print("\nProductUI_Model saved -> src/model_artifacts/ProductUI_Model")


_ilp_call_counter = [0]

def select_team_ilp(pred_df: pd.DataFrame, problem_name: str = "Dream11") -> pd.DataFrame:

    pred_df = pred_df.reset_index(drop=True).copy()
    n       = len(pred_df)

    _ilp_call_counter[0] += 1
    uid = _ilp_call_counter[0]

    prob = LpProblem(f"{problem_name}_{uid}", LpMaximize)
    x    = [LpVariable(f"x_{uid}_{i}", cat='Binary') for i in range(n)]


    prob += lpSum(pred_df['predicted_fp'].iloc[i] * x[i] for i in range(n))


    prob += lpSum(x) == 11

    for role in ROLES:
        idx = pred_df.index[pred_df['role'] == role].tolist()
        if idx:
            prob += lpSum(x[i] for i in idx) >= 1
            prob += lpSum(x[i] for i in idx) <= 8

    for team in pred_df['team'].unique():
        idx = pred_df.index[pred_df['team'] == team].tolist()
        if idx:
            prob += lpSum(x[i] for i in idx) >= 1

    prob.solve(PULP_CBC_CMD(msg=0))

    selected = pred_df[[value(x[i]) == 1 for i in range(n)]].copy()
    return selected.sort_values('predicted_fp', ascending=False).reset_index(drop=True)


def get_confidence(career_matches: int) -> str:
    if career_matches < 5:  return 'Low'
    if career_matches < 15: return 'Medium'
    return 'High'


!pip install gtts groq -q

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

import sys
if 'AI_part' in sys.modules:
    del sys.modules['AI_part']

sys.path.append(f'{BASE}/src')
from ai_part_final import compute_shap_for_squad, generate_all_commentaries, generate_team_audio

def predict_team(squad_df, team1="Team 1", team2="Team 2", audio_save_path=f"{BASE}/output/team_summary.mp3",generate_ai=False):

    squad_df = squad_df.reset_index(drop=True).copy()

    with open(f'{BASE}/src/model_artifacts/ProductUI_Model', 'rb') as f:
        artifact = pickle.load(f)

    models_loaded = artifact['models']
    feature_cols  = artifact['feature_cols']


    if 'WK' not in squad_df['role'].values:
        bat_rows = squad_df[squad_df['role'] == 'BAT']
        if not bat_rows.empty:
            squad_df.loc[bat_rows.index[0], 'role'] = 'WK'

    squad_df = compute_shap_for_squad(models_loaded, squad_df, feature_cols)


    predictions = []
    for _, row in squad_df.iterrows():
        role  = row['role']
        model = models_loaded.get(role, models_loaded['BAT'])

        X            = row[feature_cols].fillna(0).values.reshape(1, -1)
        predicted_fp = float(model.predict(X)[0])
        predicted_fp = max(0.0, round(predicted_fp, 2))

        predictions.append({
            'player':       row['player'],
            'team':         row['team'],
            'role':         role,
            'predicted_fp': predicted_fp,
            'confidence':   get_confidence(int(row.get('career_matches', 0))),
            'shap_top5':    row['shap_top5']
        })

    pred_df  = pd.DataFrame(predictions)
    selected = select_team_ilp(pred_df, problem_name="predict")


    selected['captain'] = False
    selected['vc']      = False
    selected.loc[0, 'captain'] = True
    selected.loc[1, 'vc']      = True

    selected['display_fp'] = selected['predicted_fp']
    selected.loc[selected['captain'], 'display_fp'] *= 2.0
    selected.loc[selected['vc'],      'display_fp'] *= 1.5

    if(generate_ai):
        selected = generate_all_commentaries(selected)

        audio_path = generate_team_audio(selected, team1, team2, save_path=audio_save_path)

    else:
        selected['commentary'] = ""
        audio_path = None


    return selected, audio_path



def get_actual_dream_team(match_df: pd.DataFrame) -> pd.DataFrame:

    match_df = match_df.reset_index(drop=True).copy()
    eval_df  = match_df[['player', 'team', 'role']].copy()
    eval_df['predicted_fp'] = match_df['fantasy_points'].values
    return select_team_ilp(eval_df, problem_name="actual")



print("\nEvaluating on 20 test matches...")
match_ids = test_df['match_id'].unique()[:20]
mae_list  = []

for match_id in match_ids:
    match_df = test_df[test_df['match_id'] == match_id].copy().reset_index(drop=True)

    if len(match_df) < 11:
        print(f"  Match {match_id} skipped — only {len(match_df)} players")
        continue

    team1 = match_df['team'].unique()[0]
    team2 = match_df['team'].unique()[1]

    pred_team, audio_path = predict_team(match_df, team1="Team A", team2="Team B", generate_ai=False)


    actual_team = get_actual_dream_team(match_df)


    fp_map      = match_df.set_index('player')['fantasy_points'].to_dict()
    pred_total  = sum(fp_map.get(p, 0) for p in pred_team['player'])
    dream_total = actual_team['predicted_fp'].sum()


    assert dream_total <= match_df['fantasy_points'].sum() + 1, \
        f"Match {match_id}: impossible dream total {dream_total} — bug in ILP"

    mae = abs(dream_total - pred_total)
    mae_list.append(mae)

    print(f"  Match {match_id}  |  Dream: {dream_total:.1f}  |  Predicted: {pred_total:.1f}  |  MAE: {mae:.1f}")

print(f"\nMean MAE : {np.mean(mae_list):.2f}")
print(f"Std  MAE : {np.std(mae_list):.2f}")


print("\n" + "=" * 60)
print("SMOKE TEST — First test match")
print("=" * 60)

sample_match = test_df[test_df['match_id'] == test_df['match_id'].iloc[0]].copy().reset_index(drop=True)

team1 = sample_match['team'].unique()[0]
team2 = sample_match['team'].unique()[1]
pred_team, audio_path = predict_team(sample_match, team1=team1, team2=team2, generate_ai=True)


actual_team = get_actual_dream_team(sample_match)

fp_map      = sample_match.set_index('player')['fantasy_points'].to_dict()
pred_total  = sum(fp_map.get(p, 0) for p in pred_team['player'])
dream_total = actual_team['predicted_fp'].sum()

print(f"\nDream Team total  : {dream_total:.1f}")
print(f"Your team total   : {pred_total:.1f}")
print(f"Gap (MAE)         : {abs(dream_total - pred_total):.1f}")

print("\nYour predicted team:")

for _, row in pred_team.iterrows():
    tag = " [C]" if row['captain'] else (" [VC]" if row['vc'] else "")
    print(f"\n{row['player']}{tag} | {row['role']} | {row['team']}")
    print(f"  Predicted : {row['predicted_fp']:.1f} pts | Confidence: {row['confidence']}")
    print(f"  AI Insight: {row['commentary']}")

print(f"\nAudio saved to: {audio_path}")


print("\nActual dream team:")
actual_team['actual_fp'] = actual_team['predicted_fp']
print(actual_team[['player', 'team', 'role', 'actual_fp']].to_string(index=False))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
local_train : 366,723 rows
local_val   : 10,806 rows
train_full  : 377,529 rows
test        : 103,362 rows

Validation MAEs (local split):
  BAT   MAE: 19.07
  BOWL  MAE: 18.81
  AR    MAE: 24.35
  WK    MAE: 19.06
  Weighted MAE: 20.88

Training final models on full training set:
  BAT   trained on 122,030 rows
  BOWL  trained on 82,398 rows
  AR    trained on 131,487 rows
  WK    trained on 41,614 rows

Top 10 features per role:

BAT:
            feature  importance
 venue_high_scoring    0.927645
     format_encoded    0.038088
venue_format_avg_fp    0.003705
      career_avg_fp    0.003483
     fantasy_avg_10    0.001757
        runs_avg_10    0.001224
     economy_avg_10    0.001065
      economy_std_5    0.001041
     career_matches    0.000975
      economy_avg_3    0.000933

BOWL:
             feature  importance
  venue_high_scoring    0.858163
     

/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X            = row[feature_cols].fillna(0).values.reshape(1, -1)
/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X            = row[feature_cols].fillna(0).values.reshape(1, -1)
/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('f

  Match 1519136  |  Dream: 870.0  |  Predicted: 865.0  |  MAE: 5.0


/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X            = row[feature_cols].fillna(0).values.reshape(1, -1)
/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X            = row[feature_cols].fillna(0).values.reshape(1, -1)
/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('f

  Match 1519137  |  Dream: 770.0  |  Predicted: 770.0  |  MAE: 0.0


...

  Match 1495280  |  Dream: 1704.0  |  Predicted: 1269.0  |  MAE: 435.0


/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X            = row[feature_cols].fillna(0).values.reshape(1, -1)
/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X            = row[feature_cols].fillna(0).values.reshape(1, -1)
/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('f

  Match 1495284  |  Dream: 1719.0  |  Predicted: 1525.0  |  MAE: 194.0

Mean MAE : 76.70
Std  MAE : 122.12

SMOKE TEST — First test match


/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X            = row[feature_cols].fillna(0).values.reshape(1, -1)
/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X            = row[feature_cols].fillna(0).values.reshape(1, -1)
/tmp/ipykernel_2775/10351310.py:227: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('f


Dream Team total  : 870.0
Your team total   : 865.0
Gap (MAE)         : 5.0

Your predicted team:

HT Tector [C] | BAT | Ireland
  Predicted : 83.4 pts | Confidence: High
  AI Insight: HT Tector has an expected fantasy score of 83.4 points, which receives a significant boost from his experience at high-scoring venues, elevating his score by 41.25 points. However, format-specific encoding and historical performance metrics temper expectations, deducting 14.69 points from his score. Overall, Tector's extensive career experience and performance history justify captaincy, making him a strong choice for 2x points.

G Stewart [VC] | AR | Italy
  Predicted : 81.7 pts | Confidence: High
  AI Insight: G Stewart has a 29.36-point boost on high-scoring venues, a crucial factor at Italy's upcoming match. However, his format-encoded score and average fantasy points in this format will incur a 16.16-point penalty, totaling a reduction of 11.83 points. Despite this, his experience advantage and fami